# Faster R-CNN Garbage Detection Benchmark

Notebook độc lập để huấn luyện và benchmark Faster R-CNN (ResNet-50 FPN V2). Hãy gắn dataset YOLO bằng **Add Input** trước khi chạy. Checkpoint được lưu sau mỗi epoch để hỗ trợ resume.

In [1]:
!pip install -q roboflow torchmetrics

import os
import time
import json
import random
import cv2
import numpy as np
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from roboflow import Roboflow

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {device}")

WORKING_DIR = "/kaggle/working"
DATASET_DIR = os.path.join(WORKING_DIR, "garbage_dataset")

rf = Roboflow(api_key="IPHlWsiaOStrpjGTzb7j")
project = rf.workspace("material-identification").project("garbage-classification-3")
version = project.version(2)
dataset = version.download("yolov8", location=DATASET_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 94.6 MB/s eta 0:00:00
Sử dụng thiết bị: cuda
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/garbage_dataset in yolov8:: 100%|██████████| 20940/20940 [00:01<00:00, 11666.49it/s]


In [2]:
import warnings
import yaml
import pandas as pd

MODEL_SLUG = "faster_rcnn_resnet50_fpn_v2"
MODEL_NAME = "Faster R-CNN (ResNet-50 FPN V2)"
NUM_EPOCHS = 15
BATCH_SIZE = 4
LEARNING_RATE = 0.005
VALIDATE_EVERY = 5
USE_AMP = True
USE_PRETRAINED_COCO = True
RESUME_TRAINING = True
SCORE_THRESHOLD = 0.50
IOU_THRESHOLD = 0.50

CHECKPOINT_DIR = os.path.join(WORKING_DIR, "checkpoints", MODEL_SLUG)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
LAST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "last_checkpoint.pth")
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pth")

class KaggleGarbageDataset(Dataset):
    def __init__(self, split_dir):
        self.img_dir = os.path.join(split_dir, "images")
        self.label_dir = os.path.join(split_dir, "labels")
        self.img_files = sorted([
            filename for filename in os.listdir(self.img_dir)
            if filename.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_name = self.img_files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        label_path = os.path.join(
            self.label_dir, os.path.splitext(img_name)[0] + ".txt"
        )

        image = cv2.imread(img_path)
        if image is None:
            raise ValueError(f"Cannot read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        height, width = image.shape[:2]

        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path, "r", encoding="utf-8") as label_file:
                for line in label_file:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls_id, xc, yc, bw, bh = map(float, parts)
                    xmin = max(0.0, (xc - bw / 2) * width)
                    ymin = max(0.0, (yc - bh / 2) * height)
                    xmax = min(float(width), (xc + bw / 2) * width)
                    ymax = min(float(height), (yc + bh / 2) * height)
                    if xmax > xmin and ymax > ymin:
                        boxes.append([xmin, ymin, xmax, ymax])
                        labels.append(int(cls_id) + 1)

        boxes = (
            torch.tensor(boxes, dtype=torch.float32)
            if boxes else torch.zeros((0, 4), dtype=torch.float32)
        )
        labels = torch.tensor(labels, dtype=torch.int64)
        area = (
            (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
            if len(boxes) else torch.zeros((0,), dtype=torch.float32)
        )
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor(idx, dtype=torch.int64),
            "area": area,
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
        }
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

with open(os.path.join(DATASET_DIR, "data.yaml"), "r", encoding="utf-8") as file:
    data_info = yaml.safe_load(file)

CLASS_NAMES = data_info["names"]
NUM_OBJECT_CLASSES = len(CLASS_NAMES)
NUM_CLASSES = NUM_OBJECT_CLASSES + 1

train_dataset = KaggleGarbageDataset(os.path.join(DATASET_DIR, "train"))
valid_dataset = KaggleGarbageDataset(os.path.join(DATASET_DIR, "valid"))
test_dataset = KaggleGarbageDataset(os.path.join(DATASET_DIR, "test"))

if min(len(train_dataset), len(valid_dataset), len(test_dataset)) == 0:
    raise ValueError("Train, validation và test đều phải có ít nhất một ảnh.")

loader_options = {
    "collate_fn": collate_fn,
    "num_workers": 2,
    "pin_memory": device.type == "cuda",
}
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_options
)
valid_loader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_options
)
test_loader = DataLoader(
    test_dataset, batch_size=1, shuffle=False, **loader_options
)

print(f"Classes: {CLASS_NAMES}")
print(
    f"Dataset size: train={len(train_dataset)}, "
    f"valid={len(valid_dataset)}, test={len(test_dataset)}"
)

Classes: ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']
Dataset size: train=7324, valid=2098, test=1042


In [3]:
def load_compatible_coco_weights(model, weights):
    if not USE_PRETRAINED_COCO:
        print("Pretrained COCO weights disabled.")
        return model
    try:
        pretrained_state = weights.get_state_dict(progress=True)
    except Exception as exc:
        warnings.warn(
            f"Cannot load pretrained COCO weights: {exc}. "
            "Training will start from random initialization."
        )
        return model

    model_state = model.state_dict()
    compatible_state = {
        key: value
        for key, value in pretrained_state.items()
        if key in model_state and model_state[key].shape == value.shape
    }
    model.load_state_dict(compatible_state, strict=False)
    print(
        f"Loaded {len(compatible_state)}/{len(pretrained_state)} "
        "compatible pretrained tensors."
    )
    return model

from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights

def create_model(num_classes):
    model = fasterrcnn_resnet50_fpn_v2(
        weights=None, weights_backbone=None, num_classes=num_classes
    )
    return load_compatible_coco_weights(
        model, FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    )

model = create_model(NUM_CLASSES).to(device)
print(f"{MODEL_NAME}: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


100%|██████████| 167M/167M [00:00<00:00, 319MB/s]


Loaded 400/404 compatible pretrained tensors.
Faster R-CNN (ResNet-50 FPN V2): 43.28M parameters


## Training

Cấu hình mặc định: 15 epochs, AMP trên GPU, validation mỗi 5 epochs và checkpoint sau từng epoch.

In [4]:
def load_torch_file(file_path, map_location):
    try:
        return torch.load(file_path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(file_path, map_location=map_location)

def validation_map(model, loader):
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    model.eval()
    amp_enabled = USE_AMP and device.type == "cuda"
    with torch.inference_mode():
        for images, targets in loader:
            images = [image.to(device, non_blocking=True) for image in images]
            with torch.cuda.amp.autocast(enabled=amp_enabled):
                outputs = model(images)
            predictions = [
                {key: value.cpu() for key, value in output.items()}
                for output in outputs
            ]
            ground_truth = [
                {key: value.cpu() for key, value in target.items()}
                for target in targets
            ]
            metric.update(predictions, ground_truth)
    return float(metric.compute()["map"])

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE,
    momentum=0.9,
    weight_decay=0.0005,
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=5, gamma=0.1
)
amp_enabled = USE_AMP and device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

start_epoch = 0
best_val_map = -1.0
history = []

if RESUME_TRAINING and os.path.exists(LAST_CHECKPOINT_PATH):
    checkpoint = load_torch_file(LAST_CHECKPOINT_PATH, device)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    if "scaler_state" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_map = checkpoint.get("best_val_map", -1.0)
    history = checkpoint.get("history", [])
    print(f"Resuming from epoch {start_epoch + 1}")

training_start = time.perf_counter()
for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for images, targets in train_loader:
        images = [image.to(device, non_blocking=True) for image in images]
        targets = [
            {key: value.to(device, non_blocking=True) for key, value in target.items()}
            for target in targets
        ]

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=amp_enabled):
            loss_dict = model(images, targets)
            loss = sum(loss_value for loss_value in loss_dict.values())

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss: {loss_dict}")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += float(loss.item())

    scheduler.step()
    epoch_result = {
        "epoch": epoch + 1,
        "train_loss": running_loss / max(len(train_loader), 1),
        "learning_rate": scheduler.get_last_lr()[0],
    }

    should_validate = (
        (epoch + 1) % VALIDATE_EVERY == 0
        or epoch + 1 == NUM_EPOCHS
    )
    if should_validate:
        epoch_result["val_mAP_50_95"] = validation_map(model, valid_loader)
        if epoch_result["val_mAP_50_95"] > best_val_map:
            best_val_map = epoch_result["val_mAP_50_95"]
            torch.save(model.state_dict(), BEST_MODEL_PATH)

    history.append(epoch_result)
    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "best_val_map": best_val_map,
            "history": history,
        },
        LAST_CHECKPOINT_PATH,
    )
    pd.DataFrame(history).to_csv(
        os.path.join(WORKING_DIR, f"{MODEL_SLUG}_training_history.csv"),
        index=False,
    )
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}: {epoch_result}")

training_seconds_this_run = time.perf_counter() - training_start
if not os.path.exists(BEST_MODEL_PATH):
    torch.save(model.state_dict(), BEST_MODEL_PATH)
model.load_state_dict(load_torch_file(BEST_MODEL_PATH, device))
print(f"Best validation mAP@0.50:0.95 = {best_val_map:.4f}")

`torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


Epoch 1/15: {'epoch': 1, 'train_loss': 0.4667664951274038, 'learning_rate': 0.005}
Epoch 2/15: {'epoch': 2, 'train_loss': 0.3735772123766818, 'learning_rate': 0.005}
Epoch 3/15: {'epoch': 3, 'train_loss': 0.33632878843819036, 'learning_rate': 0.005}
Epoch 4/15: {'epoch': 4, 'train_loss': 0.30940524486449494, 'learning_rate': 0.005}


`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


Epoch 5/15: {'epoch': 5, 'train_loss': 0.2886139357443197, 'learning_rate': 0.0005, 'val_mAP_50_95': 0.27752235531806946}
Epoch 6/15: {'epoch': 6, 'train_loss': 0.22439551289162304, 'learning_rate': 0.0005}
Epoch 7/15: {'epoch': 7, 'train_loss': 0.20251815202063564, 'learning_rate': 0.0005}
Epoch 8/15: {'epoch': 8, 'train_loss': 0.19076432692164738, 'learning_rate': 0.0005}
Epoch 9/15: {'epoch': 9, 'train_loss': 0.18191478804266564, 'learning_rate': 0.0005}
Epoch 10/15: {'epoch': 10, 'train_loss': 0.1750086303637579, 'learning_rate': 5e-05, 'val_mAP_50_95': 0.32469356060028076}
Epoch 11/15: {'epoch': 11, 'train_loss': 0.16443882971098683, 'learning_rate': 5e-05}
Epoch 12/15: {'epoch': 12, 'train_loss': 0.16345740847059656, 'learning_rate': 5e-05}
Epoch 13/15: {'epoch': 13, 'train_loss': 0.16254819807627532, 'learning_rate': 5e-05}
Epoch 14/15: {'epoch': 14, 'train_loss': 0.1625547756627609, 'learning_rate': 5e-05}
Epoch 15/15: {'epoch': 15, 'train_loss': 0.16051683559655688, 'learning_

## Test benchmark

Các metric: mean matched IoU, Precision, Recall, F1, AP từng lớp, mAP, latency, FPS, số tham số và kích thước checkpoint.

In [5]:
def box_iou_one(box, boxes):
    if boxes.numel() == 0:
        return torch.zeros((0,), dtype=torch.float32)
    x1 = torch.maximum(box[0], boxes[:, 0])
    y1 = torch.maximum(box[1], boxes[:, 1])
    x2 = torch.minimum(box[2], boxes[:, 2])
    y2 = torch.minimum(box[3], boxes[:, 3])
    intersection = (
        (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    )
    area1 = (
        (box[2] - box[0]).clamp(min=0)
        * (box[3] - box[1]).clamp(min=0)
    )
    area2 = (
        (boxes[:, 2] - boxes[:, 0]).clamp(min=0)
        * (boxes[:, 3] - boxes[:, 1]).clamp(min=0)
    )
    return intersection / (area1 + area2 - intersection).clamp(min=1e-8)

def detection_counts(
    prediction,
    target,
    iou_threshold=IOU_THRESHOLD,
    score_threshold=SCORE_THRESHOLD,
):
    keep = prediction["scores"] >= score_threshold
    pred_boxes = prediction["boxes"][keep].cpu()
    pred_labels = prediction["labels"][keep].cpu()
    scores = prediction["scores"][keep].cpu()
    gt_boxes = target["boxes"].cpu()
    gt_labels = target["labels"].cpu()

    matched_gt = set()
    true_positive = 0
    matched_iou_sum = 0.0

    for pred_idx in torch.argsort(scores, descending=True).tolist():
        candidates = [
            gt_idx
            for gt_idx in range(len(gt_boxes))
            if gt_idx not in matched_gt
            and int(gt_labels[gt_idx]) == int(pred_labels[pred_idx])
        ]
        if not candidates:
            continue
        ious = box_iou_one(pred_boxes[pred_idx], gt_boxes[candidates])
        best_position = int(torch.argmax(ious))
        if float(ious[best_position]) >= iou_threshold:
            matched_gt.add(candidates[best_position])
            true_positive += 1
            matched_iou_sum += float(ious[best_position])

    false_positive = len(pred_boxes) - true_positive
    false_negative = len(gt_boxes) - len(matched_gt)
    return true_positive, false_positive, false_negative, matched_iou_sum

def class_name_from_label(label_id):
    dataset_id = int(label_id) - 1
    if isinstance(CLASS_NAMES, dict):
        return str(
            CLASS_NAMES.get(
                dataset_id,
                CLASS_NAMES.get(str(dataset_id), f"class_{dataset_id}"),
            )
        )
    if 0 <= dataset_id < len(CLASS_NAMES):
        return str(CLASS_NAMES[dataset_id])
    return f"class_{dataset_id}"

def benchmark_model(model):
    coco_metric = MeanAveragePrecision(
        box_format="xyxy", iou_type="bbox", class_metrics=True
    )
    ap50_metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        iou_thresholds=[0.50],
        class_metrics=True,
    )

    model.eval()
    latencies = []
    tp = fp = fn = 0
    matched_iou_sum = 0.0
    amp_enabled = USE_AMP and device.type == "cuda"

    first_batch = next(iter(test_loader))
    warmup_images = [
        image.to(device, non_blocking=True) for image in first_batch[0]
    ]
    with torch.inference_mode():
        for _ in range(3):
            with torch.cuda.amp.autocast(enabled=amp_enabled):
                model(warmup_images)
    if device.type == "cuda":
        torch.cuda.synchronize()

    with torch.inference_mode():
        for images, targets in test_loader:
            images_device = [
                image.to(device, non_blocking=True) for image in images
            ]
            if device.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            with torch.cuda.amp.autocast(enabled=amp_enabled):
                outputs = model(images_device)
            if device.type == "cuda":
                torch.cuda.synchronize()
            elapsed_ms = (time.perf_counter() - start) * 1000.0
            latencies.extend([elapsed_ms / len(images)] * len(images))

            predictions = [
                {key: value.cpu() for key, value in output.items()}
                for output in outputs
            ]
            ground_truth = [
                {key: value.cpu() for key, value in target.items()}
                for target in targets
            ]
            coco_metric.update(predictions, ground_truth)
            ap50_metric.update(predictions, ground_truth)

            for prediction, target in zip(predictions, ground_truth):
                a, b, c, d = detection_counts(prediction, target)
                tp += a
                fp += b
                fn += c
                matched_iou_sum += d

    coco = coco_metric.compute()
    ap50 = ap50_metric.compute()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1_score = (
        2 * precision * recall / max(precision + recall, 1e-12)
    )
    average_latency = float(np.mean(latencies))

    ap_per_class = {
        class_name_from_label(class_id): float(ap)
        for class_id, ap in zip(
            coco["classes"].tolist(), coco["map_per_class"].tolist()
        )
    }
    ap50_per_class = {
        class_name_from_label(class_id): float(ap)
        for class_id, ap in zip(
            ap50["classes"].tolist(), ap50["map_per_class"].tolist()
        )
    }

    result = {
        "Model": MODEL_NAME,
        "IoU_threshold": IOU_THRESHOLD,
        "Score_threshold": SCORE_THRESHOLD,
        "Mean_IoU_matched": matched_iou_sum / max(tp, 1),
        "Precision": precision,
        "Recall": recall,
        "F1_score": f1_score,
        "AP_per_class_50_95": ap_per_class,
        "AP_per_class_50": ap50_per_class,
        "mAP_50_95": float(coco["map"]),
        "mAP_50": float(coco["map_50"]),
        "mAP_75": float(coco["map_75"]),
        "Latency_ms_per_image": average_latency,
        "FPS": 1000.0 / max(average_latency, 1e-8),
        "Parameters_M": sum(p.numel() for p in model.parameters()) / 1e6,
        "Checkpoint_MB": os.path.getsize(BEST_MODEL_PATH) / (1024 ** 2),
        "Best_validation_mAP_50_95": best_val_map,
        "Training_minutes_this_run": training_seconds_this_run / 60.0,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Num_test_images": len(test_dataset),
    }
    return result

benchmark_result = benchmark_model(model)
with open(
    os.path.join(WORKING_DIR, f"{MODEL_SLUG}_benchmark.json"),
    "w",
    encoding="utf-8",
) as file:
    json.dump(benchmark_result, file, indent=4, ensure_ascii=False)

summary_columns = [
    "Model", "Mean_IoU_matched", "Precision", "Recall", "F1_score",
    "mAP_50_95", "mAP_50", "mAP_75", "Latency_ms_per_image", "FPS",
    "Parameters_M", "Checkpoint_MB", "Best_validation_mAP_50_95",
]
display(pd.DataFrame([benchmark_result])[summary_columns].round(4))

per_class_df = pd.DataFrame([
    {
        "Class": class_name,
        "AP_50_95": ap_value,
        "AP_50": benchmark_result["AP_per_class_50"].get(class_name, -1.0),
    }
    for class_name, ap_value
    in benchmark_result["AP_per_class_50_95"].items()
])
display(per_class_df.round(4))
per_class_df.to_csv(
    os.path.join(WORKING_DIR, f"{MODEL_SLUG}_ap_per_class.csv"),
    index=False,
)
print(json.dumps(benchmark_result, indent=2, ensure_ascii=False))

`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


,Model,Mean_IoU_matched,Precision,Recall,F1_score,mAP_50_95,mAP_50,mAP_75,Latency_ms_per_image,FPS,Parameters_M,Checkpoint_MB,Best_validation_mAP_50_95
0,Faster R-CNN (ResNet-50 FPN V2),0.8667,0.6354,0.5598,0.5952,0.2918,0.4063,0.308,58.5261,17.0864,43.2818,165.4699,0.3309


,Class,AP_50_95,AP_50
0,BIODEGRADABLE,0.0571,0.0927
1,CARDBOARD,0.0570,0.0728
2,GLASS,-1.0000,-1.0000
3,METAL,0.5365,0.6906
4,PAPER,0.3977,0.5571
5,PLASTIC,0.4104,0.6182


{
  "Model": "Faster R-CNN (ResNet-50 FPN V2)",
  "IoU_threshold": 0.5,
  "Score_threshold": 0.5,
  "Mean_IoU_matched": 0.8667122062553494,
  "Precision": 0.6354365838113448,
  "Recall": 0.5597978663672094,
  "F1_score": 0.5952238805970149,
  "AP_per_class_50_95": {
    "BIODEGRADABLE": 0.05713701248168945,
    "CARDBOARD": 0.05704673007130623,
    "GLASS": -1.0,
    "METAL": 0.5364553332328796,
    "PAPER": 0.3977460563182831,
    "PLASTIC": 0.4103666841983795
  },
  "AP_per_class_50": {
    "BIODEGRADABLE": 0.09267857670783997,
    "CARDBOARD": 0.07276862114667892,
    "GLASS": -1.0,
    "METAL": 0.6905919909477234,
    "PAPER": 0.5571189522743225,
    "PLASTIC": 0.6181888580322266
  },
  "mAP_50_95": 0.29175037145614624,
  "mAP_50": 0.4062694013118744,
  "mAP_75": 0.3079872131347656,
  "Latency_ms_per_image": 58.52609265927366,
  "FPS": 17.086396076734275,
  "Parameters_M": 43.281778,
  "Checkpoint_MB": 165.4698839187622,
  "Best_validation_mAP_50_95": 0.3308511972427368,
  "Trainin